# Классификация грибов

Демонстрация обработки категориальных данных в Cart Scope на
[Mushrooms](https://www.openml.org/d/24).
Цель `class`: `e` – съедобный гриб, `p` – ядовитый.

## Содержание

1. [Подготовка](#setup)
2. [Данные](#data)
3. [Разделение выборки](#split)
4. [Редкие значения odor](#features)
5. [Конвейер](#pipeline)
6. [Обучение и оценка](#evaluation)
7. [Сохранение модели](#saving)
8. [Выводы](#conclusions)

<a id="setup"></a>
## Подготовка

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import arff
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.validation import check_is_fitted
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data/raw/mushroom.arff"
MODEL_PATH = ROOT / "models/mushroom.pkl"

<a id="data"></a>
## Данные

Загружаем ARFF, декодируем строковые значения и заменяем `?` на пропуски.
Все исходные признаки категориальные.

In [ ]:
records, metadata = arff.loadarff(DATA_PATH)
df = pd.DataFrame(records)
for column in df.columns:
    df[column] = df[column].map(
        lambda value: value.decode("utf-8") if isinstance(value, bytes) else value
    )
df = df.replace("?", np.nan)
print("Размер:", df.shape)
display(df.head())
display(pd.DataFrame({"Тип": df.dtypes.astype(str), "Пропуски": df.isna().sum()}))
display(df["class"].value_counts().rename("Количество"))

В `stalk-root` есть естественные пропуски, поэтому искусственные не добавляем.
Исходный ARFF остаётся без изменений.

<a id="split"></a>
## Разделение выборки

80% строк оставляем для обучения, 20% – для оценки, сохраняя доли классов.
Ядовитый гриб обозначаем единицей, съедобный – нулём.

In [ ]:
X = df.drop(columns="class")
y = df["class"].map({"e": 0, "p": 1})
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y,
)
print("Обучающая выборка:", X_train.shape)
print("Тестовая выборка:", X_test.shape)

<a id="features"></a>
## Редкие значения odor

Редкими считаем значения с частотой строго меньше 10% в обучающей выборке.
Каждое значение `odor` состоит из одного символа, поэтому `odor_rare_count`
равен 0 или 1. Частоты определяются в `fit`, без использования теста.
Исходный `odor` сохраняется.

In [ ]:
class RareOdorCounter(TransformerMixin, BaseEstimator):
    def __init__(self, threshold=0.1):
        self.threshold = threshold

    def fit(self, X, y=None):
        frequencies = X["odor"].value_counts(normalize=True)
        self.rare_symbols_ = frequencies[frequencies < self.threshold].index
        return self

    def transform(self, X):
        check_is_fitted(self, "rare_symbols_")
        result = X.copy()
        result["odor_rare_count"] = result["odor"].isin(self.rare_symbols_).astype(int)
        return result

<a id="pipeline"></a>
## Конвейер

Категориальные признаки: заполнение модой и OneHotEncoder.
Добавленный числовой признак: заполнение медианой и StandardScaler.
Модель – логистическая регрессия. Все преобразования включены в Pipeline.

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, ["odor_rare_count"]),
    ("categorical", categorical_pipeline, X.columns.tolist()),
])
model = Pipeline([
    ("rare_odor", RareOdorCounter()),
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])

<a id="evaluation"></a>
## Обучение и оценка

In [ ]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)
errors = int((y_pred != y_test).sum())
print(f"Accuracy: {accuracy:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"Ошибок: {errors} из {len(y_test)}")
print("Редкие значения odor:", model.named_steps["rare_odor"].rare_symbols_.tolist())

<a id="saving"></a>
## Сохранение модели

Сохраняем полный конвейер стандартным `pickle`.
Для загрузки в новом ядре сначала выполните ячейки импорта и определения `RareOdorCounter`.

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
with MODEL_PATH.open("wb") as stream:
    pickle.dump(model, stream, protocol=pickle.HIGHEST_PROTOCOL)
print("Модель сохранена:", MODEL_PATH.relative_to(ROOT))

In [ ]:
with MODEL_PATH.open("rb") as stream:
    loaded_model = pickle.load(stream)
display(pd.DataFrame({
    "Фактический класс": y_test.iloc[:5].to_numpy(),
    "Прогноз": loaded_model.predict(X_test.iloc[:5]),
    "Вероятность ядовитого класса": loaded_model.predict_proba(X_test.iloc[:5])[:, 1],
}))

<a id="conclusions"></a>
## Выводы

In [ ]:
display(Markdown(
    f"На тестовой выборке accuracy **{accuracy:.4f}**, ROC-AUC **{roc_auc:.4f}** "
    f"(округление до четырёх знаков), ошибок **{errors} из {len(y_test)}**. "
    "Конвейер объединяет создание признака, заполнение пропусков, кодирование и классификацию. "
    "Высокая оценка относится к этому набору и выбранному разбиению. "
    "Вклад признака редкости отдельно не измерялся. "
    "Для Cart Scope этот пример демонстрирует обработку категориальных данных; "
    "он не связан с прогнозом покупок."
))